In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn

In [2]:
from training.train_decoder import download_model_and_config, load_model
import einops

training decoder


In [3]:
from torch.utils.data import DataLoader
from generators.subseries_converter import EchoStateDataset
from models.vanila_decoder_transformer import TransformerDecoderModel

model_path, config = download_model_and_config('s2gk8uag')
model = load_model(model_path, config)
model.to('cuda')
model_benchmark = TransformerDecoderModel(config=config)
model_benchmark.to('cuda')
model_benchmark.eval()
config['dataset']['train']['num_series'] = 10_000
# config['dataset']['train']['series_length'] = 200
dataset_config = config.get('dataset', {})
train_dataset = EchoStateDataset(config=config, training=True, input_label=True)
val_dataset = EchoStateDataset(config=config, training=False, input_label=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  
/home/wojciech/private/magisterka/TFTS/training/train_decoder.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file

In [15]:
class Probe(nn.Module):
    def __init__(self, model, layer_name, hidden_features, out_features):
        super(Probe, self).__init__()
        self.model = model
        self.layer_name = layer_name
        self.hook = None
        self.probe = None
        self.hidden_features = hidden_features
        self.out_features = out_features

        # Find the layer and register a hook
        for name, layer in self.model.named_modules():
            if name == self.layer_name:
                self.hook = layer.register_forward_hook(self._hook_fn)
                self.probe = nn.Sequential(
                    nn.Linear(layer.out_features, self.out_features),
                    # nn.ReLU(),
                    # nn.Linear(self.hidden_features, self.out_features)
                    )
                break
        if self.hook is None:
            raise ValueError(f"Layer {self.layer_name} not found in the model.")

    def _hook_fn(self, module, input, output):
        self.activation = output

    def forward(self, x):
        _ = self.model(x[:, :-1, :])
        if hasattr(self, 'activation'):
            return self.probe(self.activation)
        else:
            raise RuntimeError("Activation not captured. Ensure the forward pass is executed.")

In [16]:
list(model.named_modules())

[('',
  TransformerDecoderModel(
    (positional_encoding): LearnablePositionalEncoding()
    (embedding): Linear(in_features=1, out_features=128, bias=True)
    (decoder_layer): TransformerDecoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (multihead_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (linear1): Linear(in_features=128, out_features=512, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=512, out_features=128, bias=True)
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (drop

In [18]:
class ProbeOptim:
    def __init__(self, underlying_model, current):
        self.probe = Probe(underlying_model, 'transformer_decoder.layers.3.linear1', hidden_features=200, out_features=1)
        self.probe.to('cuda')
        self.optimizer = torch.optim.Adam(self.probe.probe.parameters(), lr=0.01)
        self.current = current
        self.numerator = torch.zeros(1).to('cuda')
        self.denominator = torch.zeros(1).to('cuda')

    def step(self, x, y):
        output = self.probe(x)
        if self.current:
            target = y[:, 1:, :]  # Current layer
        else:
            target = y[:, :-1, :]  # Previous layer
        loss = nn.functional.mse_loss(output, target)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

    def eval(self, x, y):
        with torch.no_grad():
            output = self.probe(x)
            if self.current:
                target = y[:, 1:, :]  # Current layer
            else:
                target = y[:, :-1, :]  # Previous layer
            r2 = 1 - torch.sum((target - output) ** 2) / torch.sum((target - torch.mean(target)) ** 2)
        return r2.item() # Return quantiles of R^2

    def full_eval(self, x, y):
        with torch.no_grad():
            output = self.probe(x)
            if self.current:
                target = y[:, 1:, :]  # Current layer
            else:
                target = y[:, :-1, :]  # Previous layer
            numerator = torch.sum((target - output) ** 2)
            denominator = torch.sum((target - torch.mean(target)) ** 2)
            self.numerator += numerator
            self.denominator += denominator

    def get_full_eval_result(self):
        if self.denominator.sum() == 0:
            return None
        return 1 - (self.numerator / self.denominator)

In [19]:
import numpy as np
from tqdm import tqdm
probes = {'previous_benchmark': ProbeOptim(model_benchmark, False),
          'previous_probe': ProbeOptim(model, False)}
for epoch in range(1):
    loss = np.array([0.0])
    i = 0
    for x, y in tqdm(train_loader, total=len(train_loader)):
        x = x.to('cuda')
        y = y.float().to('cuda')
        if i % 10 == 0:
            for name, probe in probes.items():
                print(f'{name}, {probe.eval(x, y)}')
        for probe in probes.values():
            probe.step(x, y)
        i += 1

  0%|          | 0/157 [00:00<?, ?it/s]

-----------------------------------------------
Initial seed: 11000
Number of series: 10000
-----------------------------------------------


  1%|          | 1/157 [00:01<04:39,  1.79s/it]

previous_benchmark, -0.10048544406890869
previous_probe, -0.05208015441894531


  7%|▋         | 11/157 [00:13<02:53,  1.19s/it]

...

 90%|████████▉ | 141/157 [02:52<00:22,  1.38s/it]

previous_benchmark, 0.48634546995162964
previous_probe, 0.528197169303894


 96%|█████████▌| 151/157 [03:05<00:07,  1.17s/it]

previous_benchmark, 0.4815995693206787
previous_probe, 0.5353602170944214


100%|██████████| 157/157 [03:13<00:00,  1.23s/it]


In [20]:
# Ensure tensors are converted to float32 before use
numerator_cumsum = 0.0
denominator_cumsum = 0.0

with torch.no_grad():
    for x, y in tqdm(val_loader, total=len(val_loader)):
        x = x.float().to('cuda')  # Convert to float32
        y = y.float().to('cuda')  # Convert to float32
        for probe in probes.values():
            probe.full_eval(x, y)

100%|██████████| 63/63 [00:18<00:00,  3.33it/s]


In [21]:
probes['previous_benchmark'].get_full_eval_result().cpu()

tensor([0.4945])

In [22]:
probes['previous_probe'].get_full_eval_result().cpu()

tensor([0.5433])